# Order arrivals

Model in `math/07_order-arrivals.md`: orders are accepted on working days, wait for the next nesting batch and arrive
with it; the planning tool knows an order only from its arrival and reschedules completely when one arrives. The
experiments run outside the notebook (they take about an hour), with one configuration file per setting:

    uv run python -m simulation.experiment experiments/arrivals-*.json experiments/nesting-*.json --out experiments/results

This notebook reads `experiments/results/*.csv` and the plan histories in `experiments/results/<config>/plans/`.
Instance: the dental carrier example with workers (`data/generator/dental-carriers-workers.json`, **durations, worker
shares and worker counts are placeholders**), two weeks of acceptances, a backlog of 4 orders, daily nesting at 12:00,
due 4 shipping cutoffs after the acceptance, durations log-normal ($\sigma = 0.2$), start offsets ($q = 0.5$,
$\pm 5$). CP-SAT always minimizes the worker objective with cutoff slack (tardiness first, then the buffer of 60
minutes before the cutoff, overtime, breaks, activity clock) with one thread and a deterministic limit.

In [ ]:
%matplotlib inline
import csv
import json
from collections import defaultdict
from pathlib import Path
from statistics import mean, stdev

import matplotlib.pyplot as plt

from simulation.arrival_generator import generate_arrival_instance
from simulation.carrier_generator import CarrierGeneratorParams, generate_carrier_instance
from simulation.experiment import ExperimentConfig
from simulation.io.json_loader import read_model
from simulation.model.compiled import compile_instance
from simulation.model.schedule import Assignment
from simulation.objectives import worker_objective_with_slack
from simulation.planners.cpsat import solve
from simulation.viz import plot_shifts

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = ROOT / "experiments" / "results"


def rows(config):
    with (RESULTS / f"{config}.csv").open(encoding="utf-8") as f:
        return list(csv.DictReader(f))


def table(config, columns):
    by = defaultdict(list)
    for r in rows(config):
        by[r["policy"]].append(r)
    print(f"{config}  (commit {rows(config)[0]['commit']}, {len(rows(config))} runs)")
    print(f"{'policy':<22}" + "".join(f"{c:>16}" for c in columns))
    for policy, rs in by.items():
        cells = []
        for c in columns:
            values = [float(r[c]) for r in rs if r.get(c) not in (None, "")]
            cells.append(f"{mean(values):10.1f} ±{stdev(values):4.1f}" if len(values) > 1 else f"{values[0]:16.1f}")
        print(f"{policy:<22}" + "".join(cells))
    print()

## CP-SAT on a static instance of realistic size

The 40 orders of the example with workers, all known at 0 (no arrivals), solved once with the worker objective (60 s,
one thread). The gap between the best plan and the bound shows how far CP-SAT is from proving optimality at this
size (relevant for the question whether a GA is needed).

In [ ]:
static = compile_instance(generate_carrier_instance(read_model(ROOT / "data/generator/dental-carriers-workers.json",
                                                               CarrierGeneratorParams)))
result = solve(static, worker_objective_with_slack(60), time_limit=60, workers=1, deterministic_time=60)
print(f"{static.n_ops} operations: {result.status}, levels {result.levels}, bound of the last level {result.bound}")

## Load levels

Four policies; all start from a CP-SAT plan of the orders known at 0 (the hindsight policy from a plan of all orders):
- **rule**: SPT + ECT at every arrival, shift repair otherwise;
- **cpsat**: CP-SAT at every arrival, shift repair otherwise;
- **cpsat+thresholds**: also CP-SAT when the repaired plan deviates (makespan 10, start 30 minutes) or every 120 minutes;
- **hindsight+thresholds**: every arrival known from the start (a plan of the full instance), same thresholds; the gap
  to cpsat+thresholds is the cost of not knowing the arrivals (with disturbances mixed in, see the undisturbed run).

Mean ± standard deviation over 5 seeds. `tardy_days`: missed shipping cutoffs summed over orders; `overtime`:
minutes; `mean_lead_time`: minutes from acceptance to completion; `bottleneck_load`: busy share of the busiest machine
over the realized makespan (around the clock).

In [ ]:
COLUMNS = ["orders", "tardy_days", "overtime", "mean_lead_time", "reschedulings", "planner_time", "bottleneck_load"]
for config in ["arrivals-rate4", "arrivals-rate6", "arrivals-rate8", "arrivals-rate8-double"]:
    table(config, COLUMNS)

In [ ]:
levels = ["arrivals-rate4", "arrivals-rate6", "arrivals-rate8"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for policy in ["rule", "cpsat", "cpsat+thresholds", "hindsight+thresholds"]:
    for ax, column in zip(axes, ["tardy_days", "overtime"], strict=True):
        ax.plot([4, 6, 8], [mean(float(r[column]) for r in rows(c) if r["policy"] == policy) for c in levels],
                marker="o", label=policy)
        ax.set_xlabel("orders per working day")
        ax.set_title(column)
axes[0].legend()
plt.tight_layout()
plt.show()

## Cost of not knowing the arrivals (undisturbed)

$\sigma = 0$ and no start offsets: the gap between cpsat+thresholds and hindsight+thresholds is only information.
Check: if the hindsight policy is worse here, its start plan (a much larger CP-SAT problem, `start_status`) is the
likely cause, not the information; such a result is flagged, not reported.

In [ ]:
table("arrivals-rate6-undisturbed", COLUMNS)
for r in rows("arrivals-rate6-undisturbed"):
    if r["policy"] == "hindsight+thresholds":
        print(f"seed {r['seed']}: hindsight start plan {r['start_status']}, levels {r['start_levels']}")

## Daily against weekly nesting

The advisor's question: nesting only on Fridays at 12:00 against every working day, both due 6 cutoffs after the
acceptance (the shortest offset the weekly cycle allows), cpsat+thresholds, 6 orders per working day.

In [ ]:
for config in ["nesting-daily", "nesting-weekly"]:
    table(config, ["orders", "tardy_days", "overtime", "mean_lead_time", "mean_flow_time", "bottleneck_load"])

## The plan around one arrival

From the plan history of one run (`plans/<policy>-<seed>.jsonl`: the start plan, then every plan the factory receives
as a change set): the plan just before and just after the first arrival rescheduling.

In [ ]:
CONFIG, POLICY, SEED = "arrivals-rate6", "cpsat+thresholds", 1
config = read_model(ROOT / "experiments" / f"{CONFIG}.json", ExperimentConfig)
ci = compile_instance(generate_arrival_instance(config.arrivals.model_copy(update={"seed": SEED})))
history = [json.loads(line) for line in (RESULTS / CONFIG / "plans" / f"{POLICY}-{SEED}.jsonl").open()]


def plan_after(i):
    plan = {}
    for h in history[:i + 1]:
        plan |= {(c["order_id"], c["op_id"]): c for c in h["changed"]}
    return [Assignment(**c) for c in plan.values()], [b for b in history[i]["breaks"]]


i = next(i for i, h in enumerate(history) if "arrival" in h["triggers"] and h["t"] > 0)
for j, label in [(i - 1, "before"), (i, "after")]:
    plan, breaks = plan_after(j)
    plot_shifts(ci, plan, [], title=f"{label} the arrival at t = {history[i]['t']}")
    plt.show()
print(f"{len(history)} plans in the run, {sum('arrival' in h['triggers'] for h in history)} at arrivals")

**Observations:** to be written after the run.